# Cortex Code — FinOps Governance Toolkit

A three-section notebook for understanding, controlling, and monitoring Cortex Code spend across both surfaces:
- **CLI** — `CORTEX_CODE_CLI_USAGE_HISTORY` (terminal / `snow` command)
- **Snowsight** — `CORTEX_CODE_SNOWSIGHT_USAGE_HISTORY` (browser IDE)

| Section | What it does |
|---------|-------------|
| **A — Spend Analysis** | Daily trend, top users, model breakdown, projections, pricing reference |
| **B — Per-User Daily Limits** | Current settings, user overrides, rolling 24h usage vs limits, limit calculator |
| **C — Threshold Notifications** | Monthly budget alerts (native) + per-user limit approach alerts (custom task) |

**AI Credit price:** $2.00 per credit (global on-demand, April 2026 consumption table).  
Capacity customers: adjust `ai_credit_price` in the config cell below.

**Required privilege:**
```sql
GRANT IMPORTED PRIVILEGES ON DATABASE SNOWFLAKE TO ROLE <your_role>;
```

**Sections B and C require `ACCOUNTADMIN`** to read/set parameters and create notification objects.

See also: `scenarios/` for step-by-step runbooks, `worksheets/` for standalone Snowsight SQL.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()

ai_credit_price = 2.00

source = 'both'  # Options: 'cli', 'snowsight', 'both'

CLI_TABLE       = 'SNOWFLAKE.ACCOUNT_USAGE.CORTEX_CODE_CLI_USAGE_HISTORY'
SNOWSIGHT_TABLE = 'SNOWFLAKE.ACCOUNT_USAGE.CORTEX_CODE_SNOWSIGHT_USAGE_HISTORY'

if source == 'cli':
    source_table = CLI_TABLE
elif source == 'snowsight':
    source_table = SNOWSIGHT_TABLE
else:
    source_table = f"""(
        SELECT 'cli'       AS source, * FROM {CLI_TABLE}
        UNION ALL
        SELECT 'snowsight' AS source, * FROM {SNOWSIGHT_TABLE}
    ) t"""

print(f"AI Credit price : ${ai_credit_price:.2f} per credit")
print(f"Source          : {source}")
print("Adjust ai_credit_price or source above before running all cells.")

## 1. Daily Usage Summary
Request counts, tokens, credits, and estimated dollar cost per day — last 30 days.

In [ ]:
daily_usage = session.sql(f"""
    SELECT
        USAGE_TIME::DATE                        AS usage_date,
        COUNT(*)                                AS total_requests,
        SUM(TOKENS)                             AS total_tokens,
        ROUND(SUM(TOKEN_CREDITS), 4)            AS total_credits,
        ROUND(SUM(TOKEN_CREDITS) * {ai_credit_price}, 2) AS estimated_cost_usd
    FROM {source_table}
    GROUP BY 1
    ORDER BY 1 DESC
    LIMIT 30
""").to_pandas()
daily_usage

## 2. Weekly Usage Trend
Aggregates by week — useful for spotting adoption ramp and seasonal patterns.

In [ ]:
weekly_trend = session.sql(f"""
    SELECT
        DATE_TRUNC('WEEK', USAGE_TIME)          AS week_start,
        COUNT(*)                                AS total_requests,
        SUM(TOKENS)                             AS total_tokens,
        ROUND(SUM(TOKEN_CREDITS), 4)            AS total_credits,
        ROUND(SUM(TOKEN_CREDITS) * {ai_credit_price}, 2) AS estimated_cost_usd,
        ROUND(AVG(TOKENS), 0)                   AS avg_tokens_per_request
    FROM {source_table}
    GROUP BY 1
    ORDER BY 1 DESC
    LIMIT 13
""").to_pandas()
weekly_trend

## 3. Top Users by Credits
Ranks all users by total AI credit consumption, with first/last usage timestamps.

In [ ]:
top_users = session.sql(f"""
    SELECT
        USER_ID,
        COUNT(*)                             AS total_requests,
        SUM(TOKENS)                             AS total_tokens,
        ROUND(SUM(TOKEN_CREDITS), 4)            AS total_credits,
        ROUND(SUM(TOKEN_CREDITS) * {ai_credit_price}, 2) AS estimated_cost_usd,
        MIN(USAGE_TIME)                         AS first_usage,
        MAX(USAGE_TIME)                         AS last_usage
    FROM {source_table}
    GROUP BY 1
    ORDER BY total_credits DESC
    LIMIT 20
""").to_pandas()
top_users

## 4. Hourly Usage Pattern
Reveals which hours of the day see peak activity — all-time aggregate.

In [ ]:
hourly_pattern = session.sql(f"""
    SELECT
        HOUR(USAGE_TIME)                        AS hour_of_day,
        COUNT(*)                                AS total_requests,
        SUM(TOKENS)                             AS total_tokens,
        ROUND(SUM(TOKEN_CREDITS), 4)            AS total_credits
    FROM {source_table}
    GROUP BY 1
    ORDER BY 1
""").to_pandas()
hourly_pattern

## 5. Usage by Model
Breaks out credits by model using the `CREDITS_GRANULAR` variant column, with cache read/write split.

In [ ]:
usage_by_model = session.sql(f"""
    SELECT
        f.key                                                   AS model,
        COUNT(*)                                                AS total_requests,
        SUM(NVL(f.value:cache_read_input::FLOAT, 0))            AS cache_read_credits,
        SUM(NVL(f.value:cache_write_input::FLOAT, 0))           AS cache_write_credits,
        SUM(NVL(f.value:input::FLOAT, 0))                       AS input_credits,
        SUM(NVL(f.value:output::FLOAT, 0))                      AS output_credits,
        ROUND(
            SUM(NVL(f.value:cache_read_input::FLOAT, 0)
                + NVL(f.value:cache_write_input::FLOAT, 0)
                + NVL(f.value:input::FLOAT, 0)
                + NVL(f.value:output::FLOAT, 0)), 4)            AS total_credits,
        ROUND(
            SUM(NVL(f.value:cache_read_input::FLOAT, 0)
                + NVL(f.value:cache_write_input::FLOAT, 0)
                + NVL(f.value:input::FLOAT, 0)
                + NVL(f.value:output::FLOAT, 0)) * {ai_credit_price}, 2) AS estimated_cost_usd
    FROM {source_table},
        LATERAL FLATTEN(input => CREDITS_GRANULAR) f
    GROUP BY 1
    ORDER BY total_credits DESC
""").to_pandas()
usage_by_model

## 6. Cost Projections
Takes the 22 busiest working days on record (proxy for a full working month), then projects to weekly, monthly, and annual spend.

In [ ]:
cost_projections = session.sql(f"""
    WITH daily_costs AS (
        SELECT
            USAGE_TIME::DATE            AS usage_date,
            SUM(TOKEN_CREDITS)          AS daily_credits
        FROM {source_table}
        GROUP BY 1
        ORDER BY daily_credits DESC
        LIMIT 22
    ),
    stats AS (
        SELECT
            MIN(daily_credits)  AS min_daily,
            AVG(daily_credits)  AS mean_daily,
            MAX(daily_credits)  AS max_daily
        FROM daily_costs
    )
    SELECT 'Per Day'   AS period,
        ROUND(min_daily, 2)             AS min_credits,
        ROUND(mean_daily, 2)            AS mean_credits,
        ROUND(max_daily, 2)             AS max_credits,
        ROUND(min_daily * {ai_credit_price}, 2)      AS min_cost_usd,
        ROUND(mean_daily * {ai_credit_price}, 2)     AS mean_cost_usd,
        ROUND(max_daily * {ai_credit_price}, 2)      AS max_cost_usd
    FROM stats
    UNION ALL
    SELECT 'Per Week',
        ROUND(min_daily * 5, 2),  ROUND(mean_daily * 5, 2),  ROUND(max_daily * 5, 2),
        ROUND(min_daily * 5 * {ai_credit_price}, 2), ROUND(mean_daily * 5 * {ai_credit_price}, 2), ROUND(max_daily * 5 * {ai_credit_price}, 2)
    FROM stats
    UNION ALL
    SELECT 'Per Month',
        ROUND(min_daily * 22, 2), ROUND(mean_daily * 22, 2), ROUND(max_daily * 22, 2),
        ROUND(min_daily * 22 * {ai_credit_price}, 2), ROUND(mean_daily * 22 * {ai_credit_price}, 2), ROUND(max_daily * 22 * {ai_credit_price}, 2)
    FROM stats
    UNION ALL
    SELECT 'Per Year',
        ROUND(min_daily * 260, 2), ROUND(mean_daily * 260, 2), ROUND(max_daily * 260, 2),
        ROUND(min_daily * 260 * {ai_credit_price}, 2), ROUND(mean_daily * 260 * {ai_credit_price}, 2), ROUND(max_daily * 260 * {ai_credit_price}, 2)
    FROM stats
""").to_pandas()
cost_projections

## 7. CLI vs Snowsight Comparison
Side-by-side totals for each surface — always queries both sources regardless of the `source` setting above.

In [ ]:
source_comparison = session.sql(f"""
    WITH cli AS (
        SELECT
            'cli'                               AS surface,
            COUNT(*)                            AS total_requests,
            SUM(TOKENS)                         AS total_tokens,
            ROUND(SUM(TOKEN_CREDITS), 4)        AS total_credits,
            ROUND(SUM(TOKEN_CREDITS) * {ai_credit_price}, 2) AS estimated_cost_usd
        FROM {CLI_TABLE}
    ),
    snowsight AS (
        SELECT
            'snowsight'                         AS surface,
            COUNT(*)                            AS total_requests,
            SUM(TOKENS)                         AS total_tokens,
            ROUND(SUM(TOKEN_CREDITS), 4)        AS total_credits,
            ROUND(SUM(TOKEN_CREDITS) * {ai_credit_price}, 2) AS estimated_cost_usd
        FROM {SNOWSIGHT_TABLE}
    )
    SELECT * FROM cli
    UNION ALL
    SELECT * FROM snowsight
""").to_pandas()
source_comparison

## 8. Cortex Code Model Pricing Reference
Official rates from the Snowflake Service Consumption Table, **Table 6(e) — Cortex Code** (effective April 1, 2026).  
All rates are AI Credits per one million tokens. Multiply by your AI credit price to get USD cost per 1M tokens.

In [ ]:
models = [
    {"model": "claude-4-sonnet",   "input": 1.50, "output": 7.50,  "cache_write": 1.88, "cache_read": 0.15},
    {"model": "claude-opus-4-5",   "input": 2.75, "output": 13.75, "cache_write": 3.44, "cache_read": 0.28},
    {"model": "claude-opus-4-6",   "input": 2.75, "output": 13.75, "cache_write": 3.44, "cache_read": 0.28},
    {"model": "claude-sonnet-4-5", "input": 1.65, "output": 8.25,  "cache_write": 2.07, "cache_read": 0.17},
    {"model": "claude-sonnet-4-6", "input": 1.65, "output": 8.25,  "cache_write": 2.07, "cache_read": 0.17},
    {"model": "openai-gpt-5.2",    "input": 0.97, "output": 7.70,  "cache_write": None, "cache_read": 0.10},
    {"model": "openai-gpt-5.4",    "input": 1.38, "output": 8.25,  "cache_write": None, "cache_read": 0.14},
]

df = pd.DataFrame(models)
df.columns = [
    "Model",
    "Input (AI Credits/1M tokens)",
    "Output (AI Credits/1M tokens)",
    "Cache Write (AI Credits/1M tokens)",
    "Cache Read (AI Credits/1M tokens)",
]

print(f"Source: Snowflake Service Consumption Table, Table 6(e) — Cortex Code (effective April 1, 2026)")
print(f"AI Credit On-Demand Price: ${ai_credit_price:.2f}/credit (global)")
print()
df

---

# Section B: Per-User Daily Credit Limits

Snowflake provides two parameters that cap how many estimated AI credits each user can consume in a rolling 24-hour window:

| Parameter | Surface |
|-----------|---------|
| `CORTEX_CODE_CLI_DAILY_EST_CREDIT_LIMIT_PER_USER` | Cortex Code CLI |
| `CORTEX_CODE_SNOWSIGHT_DAILY_EST_CREDIT_LIMIT_PER_USER` | Cortex Code in Snowsight |

**Values:** `-1` = unlimited (default), `0` = blocked entirely, positive number = daily credit cap.  
**Scope:** Set at account level (all users) or per-user (overrides account default).  
**Enforcement:** Snowflake blocks the user when their rolling 24h usage exceeds the limit. No warning is sent — Section C adds proactive alerts.

[Snowflake docs: Cost controls for Cortex Code](https://docs.snowflake.com/en/user-guide/cortex-code/credit-usage-limit)

In [ ]:
print("=== Account-Level Daily Credit Limits ===\n")
try:
    for param in [
        'CORTEX_CODE_CLI_DAILY_EST_CREDIT_LIMIT_PER_USER',
        'CORTEX_CODE_SNOWSIGHT_DAILY_EST_CREDIT_LIMIT_PER_USER',
    ]:
        rows = session.sql(
            f"SHOW PARAMETERS LIKE '{param}' IN ACCOUNT"
        ).to_pandas()
        if rows.empty:
            print(f"{param}: not set (default = -1, unlimited)")
        else:
            val = rows.iloc[0]['value']
            level = rows.iloc[0]['level']
            label = 'unlimited' if str(val) == '-1' else f'{val} credits/day'
            print(f"{param}")
            print(f"  Value: {val} ({label})  |  Set at: {level}\n")
except Exception as e:
    print(f"Could not query account parameters (ACCOUNTADMIN required): {e}")

### B2 — Users with Per-User Limit Overrides
Lists every user whose daily credit limit has been set individually (overriding the account default). Adapted from the [Snowflake docs](https://docs.snowflake.com/en/user-guide/cortex-code/credit-usage-limit#listing-users-with-custom-limits).

In [ ]:
overrides = []
users_df = session.sql("SHOW USERS").to_pandas()

for _, row in users_df.iterrows():
    uname = row['name']
    for param in [
        'CORTEX_CODE_CLI_DAILY_EST_CREDIT_LIMIT_PER_USER',
        'CORTEX_CODE_SNOWSIGHT_DAILY_EST_CREDIT_LIMIT_PER_USER',
    ]:
        try:
            p = session.sql(
                f"SHOW PARAMETERS LIKE '{param}' IN USER \"{uname}\""
            ).to_pandas()
            user_rows = p[p['level'] == 'USER']
            if not user_rows.empty:
                overrides.append({
                    'user_name': uname,
                    'parameter': param,
                    'value': user_rows.iloc[0]['value'],
                })
        except Exception:
            pass

if overrides:
    override_df = pd.DataFrame(overrides)
    print(f"{len(override_df)} per-user override(s) found:\n")
    override_df
else:
    print("No per-user overrides found. All users inherit the account-level default.")

### B3 — Rolling 24h Usage vs. Effective Limits
Shows each active user's credit consumption over the last 24 hours alongside their effective daily limit. Users at **≥80%** of their limit are flagged — they are close to being blocked.

In [ ]:
usage_vs_limits = session.sql(f"""
    WITH user_usage AS (
        SELECT USER_ID, 'cli' AS surface,
               SUM(TOKEN_CREDITS) AS rolling_24h_credits
        FROM {CLI_TABLE}
        WHERE USAGE_TIME >= DATEADD('hour', -24, CURRENT_TIMESTAMP)
        GROUP BY USER_ID
        UNION ALL
        SELECT USER_ID, 'snowsight' AS surface,
               SUM(TOKEN_CREDITS) AS rolling_24h_credits
        FROM {SNOWSIGHT_TABLE}
        WHERE USAGE_TIME >= DATEADD('hour', -24, CURRENT_TIMESTAMP)
        GROUP BY USER_ID
    ),
    acct_defaults AS (
        SELECT
            MAX(CASE WHEN "key" ILIKE '%CLI%' THEN "value" END)       AS cli_default,
            MAX(CASE WHEN "key" ILIKE '%SNOWSIGHT%' THEN "value" END) AS ss_default
        FROM TABLE(RESULT_SCAN(
            LAST_QUERY_ID(-2)
        ))
    )
    SELECT u.NAME                                      AS user_name,
           uu.surface,
           ROUND(uu.rolling_24h_credits, 4)            AS credits_24h,
           CASE uu.surface
               WHEN 'cli'       THEN NVL(ad.cli_default, -1)
               WHEN 'snowsight' THEN NVL(ad.ss_default, -1)
           END                                         AS effective_limit,
           CASE
               WHEN NVL(
                   CASE uu.surface
                       WHEN 'cli'       THEN ad.cli_default
                       WHEN 'snowsight' THEN ad.ss_default
                   END, -1) <= 0 THEN NULL
               ELSE ROUND(
                   uu.rolling_24h_credits /
                   CASE uu.surface
                       WHEN 'cli'       THEN ad.cli_default
                       WHEN 'snowsight' THEN ad.ss_default
                   END * 100, 1)
           END                                         AS pct_of_limit
    FROM user_usage uu
    JOIN SNOWFLAKE.ACCOUNT_USAGE.USERS u ON uu.USER_ID = u.USER_ID
    CROSS JOIN acct_defaults ad
    ORDER BY uu.rolling_24h_credits DESC
""").to_pandas()

if not usage_vs_limits.empty:
    at_risk = usage_vs_limits[
        usage_vs_limits['PCT_OF_LIMIT'].notna() &
        (usage_vs_limits['PCT_OF_LIMIT'] >= 80)
    ]
    if not at_risk.empty:
        print(f"⚠ {len(at_risk)} user/surface pair(s) at ≥80% of their daily limit:\n")
        print(at_risk.to_string(index=False))
        print()
    usage_vs_limits
else:
    print("No Cortex Code usage in the last 24 hours.")

### B4 — Limit Calculator
Uses your actual per-user daily spend patterns to recommend a daily credit limit. Shows how many users would be affected at each candidate limit, then generates ready-to-run `ALTER` statements.

Adjust `proposed_limit` below to explore different thresholds.

In [ ]:
proposed_limit = 10  # credits per day — adjust this value

impact = session.sql(f"""
    WITH daily_per_user AS (
        SELECT USER_ID,
               USAGE_TIME::DATE AS usage_date,
               SUM(TOKEN_CREDITS) AS daily_credits
        FROM (
            SELECT USER_ID, USAGE_TIME, TOKEN_CREDITS FROM {CLI_TABLE}
            UNION ALL
            SELECT USER_ID, USAGE_TIME, TOKEN_CREDITS FROM {SNOWSIGHT_TABLE}
        )
        WHERE USAGE_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP)
        GROUP BY USER_ID, USAGE_TIME::DATE
    ),
    user_stats AS (
        SELECT USER_ID,
               COUNT(*)             AS active_days,
               ROUND(AVG(daily_credits), 4)  AS avg_daily,
               ROUND(MAX(daily_credits), 4)  AS max_daily,
               ROUND(PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY daily_credits), 4) AS p90_daily
        FROM daily_per_user
        GROUP BY USER_ID
    )
    SELECT u.NAME                                   AS user_name,
           s.active_days,
           s.avg_daily,
           s.p90_daily,
           s.max_daily,
           CASE WHEN s.max_daily > {proposed_limit} THEN 'YES' ELSE 'no' END AS would_hit_limit
    FROM user_stats s
    JOIN SNOWFLAKE.ACCOUNT_USAGE.USERS u ON s.USER_ID = u.USER_ID
    ORDER BY s.max_daily DESC
""").to_pandas()

if not impact.empty:
    hit = impact[impact['WOULD_HIT_LIMIT'] == 'YES']
    total = len(impact)
    print(f"Proposed limit: {proposed_limit} credits/day")
    print(f"  {len(hit)} of {total} active users would have hit this limit at least once in the last 30 days")
    print(f"  {total - len(hit)} users would be unaffected\n")
    print(impact.to_string(index=False))
    print(f"\n--- Ready-to-run SQL ---\n")
    print(f"-- Set account-wide daily limit (both surfaces)")
    print(f"ALTER ACCOUNT SET CORTEX_CODE_CLI_DAILY_EST_CREDIT_LIMIT_PER_USER = {proposed_limit};")
    print(f"ALTER ACCOUNT SET CORTEX_CODE_SNOWSIGHT_DAILY_EST_CREDIT_LIMIT_PER_USER = {proposed_limit};")
    if not hit.empty:
        print(f"\n-- Or: set a higher limit for power users only")
        for uname in hit['USER_NAME'].tolist()[:5]:
            print(f"ALTER USER \"{uname}\" SET CORTEX_CODE_CLI_DAILY_EST_CREDIT_LIMIT_PER_USER = {proposed_limit * 3};")
            print(f"ALTER USER \"{uname}\" SET CORTEX_CODE_SNOWSIGHT_DAILY_EST_CREDIT_LIMIT_PER_USER = {proposed_limit * 3};")
else:
    print("No usage data in the last 30 days — cannot calculate limit impact.")

---

# Section C: Threshold Notifications

Two tiers of proactive alerting:

| Tier | Mechanism | What it alerts on |
|------|-----------|-------------------|
| **1 — Monthly budget** | Snowflake-native budget notification | Account or custom budget reaching a % of its monthly limit |
| **2 — Per-user daily limit** | Custom task + stored procedure (deployed below) | Individual user approaching their rolling 24h credit cap |

**Why Tier 2 matters:** Snowflake blocks users when they hit their per-user daily limit, but provides **no warning**. The task-based alert system below sends a notification when a user crosses a configurable threshold (default 80%) so they can adjust before being locked out.

### Configuration
Set these values before running the cells below.

In [ ]:
notification_email    = 'finops@example.com'       # change to your email
alert_threshold_pct   = 80                          # notify when user reaches this % of daily limit
alert_cooldown_hours  = 4                           # suppress repeat alerts for this many hours
task_schedule_minutes = 15                          # how often the alert task runs
monthly_budget_limit  = 1000                        # monthly credit limit for account budget
budget_alert_pct      = 0.80                        # budget notification threshold (0.0 - 1.0)
governance_schema     = 'SNOWFLAKE_EXAMPLE.CODE_SPEND_CONTROLS'  # schema for alert objects

print("=== Notification Configuration ===")
print(f"  Email               : {notification_email}")
print(f"  Alert threshold     : {alert_threshold_pct}% of daily limit")
print(f"  Cooldown            : {alert_cooldown_hours} hours between repeat alerts")
print(f"  Task schedule       : every {task_schedule_minutes} minutes")
print(f"  Monthly budget limit: {monthly_budget_limit} credits")
print(f"  Budget alert at     : {int(budget_alert_pct * 100)}% of monthly limit")
print(f"  Governance schema   : {governance_schema}")

### Tier 1 — Monthly Budget Notifications (Snowflake-native)
Creates an email notification integration and attaches it to the account root budget at the configured threshold. Requires `ACCOUNTADMIN`.

In [ ]:
try:
    session.sql("USE ROLE ACCOUNTADMIN").collect()

    session.sql(f"""
        CREATE NOTIFICATION INTEGRATION IF NOT EXISTS cortex_code_budget_email_int
            TYPE = EMAIL
            ENABLED = TRUE
            ALLOWED_RECIPIENTS = ('{notification_email}')
            COMMENT = 'Budget threshold alerts for Cortex Code governance'
    """).collect()
    print(f"✓ Notification integration 'cortex_code_budget_email_int' ready")

    session.sql("CALL SNOWFLAKE.LOCAL.ACCOUNT_ROOT_BUDGET!ACTIVATE()").collect()
    print("✓ Account root budget activated")

    session.sql(
        f"CALL SNOWFLAKE.LOCAL.ACCOUNT_ROOT_BUDGET!SET_SPENDING_LIMIT({monthly_budget_limit})"
    ).collect()
    print(f"✓ Monthly spending limit set to {monthly_budget_limit} credits")

    session.sql(f"""
        CALL SNOWFLAKE.LOCAL.ACCOUNT_ROOT_BUDGET!ADD_EMAIL_NOTIFICATION_INTEGRATION(
            'cortex_code_budget_email_int',
            {budget_alert_pct},
            'Alert',
            '{notification_email}'
        )
    """).collect()
    print(f"✓ Email notification at {int(budget_alert_pct * 100)}% → {notification_email}")

    print("\n=== Registered Notifications ===")
    notifs = session.sql(
        "CALL SNOWFLAKE.LOCAL.ACCOUNT_ROOT_BUDGET!GET_NOTIFICATION_INTEGRATIONS()"
    ).to_pandas()
    print(notifs.to_string(index=False) if not notifs.empty else "None registered")

except Exception as e:
    print(f"Budget notification setup failed (ACCOUNTADMIN required): {e}")

### Tier 2 — Per-User Daily Limit Approach Alerts

Snowflake blocks users at their daily credit limit but **does not warn them beforehand**. This cell deploys three objects that fill the gap:

1. **`CORTEX_CODE_LIMIT_ALERTS`** — audit table logging every alert sent
2. **`CORTEX_CODE_LIMIT_ALERT_CHECK`** — stored procedure that compares each user's rolling 24h usage against their effective limit and sends an email when usage ≥ the configured threshold %
3. **`CORTEX_CODE_LIMIT_ALERT_TASK`** — serverless task that runs the procedure on a schedule

Requires `ACCOUNTADMIN` (to read user parameters and send email). The procedure uses `SYSTEM$SEND_EMAIL` via the notification integration created in Tier 1.

In [ ]:
try:
    session.sql("USE ROLE ACCOUNTADMIN").collect()
    session.sql(f"CREATE SCHEMA IF NOT EXISTS {governance_schema}").collect()
    session.sql(f"GRANT USAGE ON DATABASE SNOWFLAKE_EXAMPLE TO APPLICATION SNOWFLAKE").collect()
    session.sql(f"GRANT USAGE ON SCHEMA {governance_schema} TO APPLICATION SNOWFLAKE").collect()

    # 1. Audit table
    session.sql(f"""
        CREATE TABLE IF NOT EXISTS {governance_schema}.CORTEX_CODE_LIMIT_ALERTS (
            user_name       VARCHAR,
            surface         VARCHAR,
            usage_credits   NUMBER(12,4),
            limit_credits   NUMBER(12,4),
            pct_used        NUMBER(5,1),
            alerted_at      TIMESTAMP_TZ DEFAULT CURRENT_TIMESTAMP
        )
    """).collect()
    print("✓ Audit table CORTEX_CODE_LIMIT_ALERTS ready")

    # 2. Stored procedure
    session.sql(f"""
        CREATE OR REPLACE PROCEDURE {governance_schema}.CORTEX_CODE_LIMIT_ALERT_CHECK()
        RETURNS VARCHAR
        LANGUAGE SQL
        EXECUTE AS OWNER
        AS
        $$
        DECLARE
            alert_threshold   FLOAT DEFAULT {alert_threshold_pct};
            cooldown_hours    INT   DEFAULT {alert_cooldown_hours};
            notif_integration VARCHAR DEFAULT 'cortex_code_budget_email_int';
            notif_email       VARCHAR DEFAULT '{notification_email}';
            cli_default       FLOAT DEFAULT -1;
            ss_default        FLOAT DEFAULT -1;
            alert_count       INT DEFAULT 0;
            c_users CURSOR FOR
                WITH user_usage AS (
                    SELECT USER_ID, 'cli' AS surface,
                           SUM(TOKEN_CREDITS) AS rolling_24h_credits
                    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_CODE_CLI_USAGE_HISTORY
                    WHERE USAGE_TIME >= DATEADD('hour', -24, CURRENT_TIMESTAMP)
                    GROUP BY USER_ID
                    UNION ALL
                    SELECT USER_ID, 'snowsight' AS surface,
                           SUM(TOKEN_CREDITS) AS rolling_24h_credits
                    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_CODE_SNOWSIGHT_USAGE_HISTORY
                    WHERE USAGE_TIME >= DATEADD('hour', -24, CURRENT_TIMESTAMP)
                    GROUP BY USER_ID
                )
                SELECT u.NAME                           AS user_name,
                       uu.surface,
                       ROUND(uu.rolling_24h_credits, 4) AS usage_credits,
                       CASE uu.surface
                           WHEN 'cli'       THEN :cli_default
                           WHEN 'snowsight' THEN :ss_default
                       END                              AS limit_credits,
                       ROUND(uu.rolling_24h_credits /
                           NULLIF(CASE uu.surface
                               WHEN 'cli'       THEN :cli_default
                               WHEN 'snowsight' THEN :ss_default
                           END, 0) * 100, 1)            AS pct_used
                FROM user_usage uu
                JOIN SNOWFLAKE.ACCOUNT_USAGE.USERS u ON uu.USER_ID = u.USER_ID
                WHERE CASE uu.surface
                          WHEN 'cli'       THEN :cli_default
                          WHEN 'snowsight' THEN :ss_default
                      END > 0
                  AND ROUND(uu.rolling_24h_credits /
                      NULLIF(CASE uu.surface
                          WHEN 'cli'       THEN :cli_default
                          WHEN 'snowsight' THEN :ss_default
                      END, 0) * 100, 1) >= :alert_threshold;
        BEGIN
            -- Read account defaults
            LET rs_cli RESULTSET := (
                SHOW PARAMETERS LIKE 'CORTEX_CODE_CLI_DAILY_EST_CREDIT_LIMIT_PER_USER' IN ACCOUNT
            );
            LET df_cli RESULTSET := (SELECT "value"::FLOAT AS v FROM TABLE(RESULT_SCAN(LAST_QUERY_ID())));
            FOR r IN df_cli DO
                cli_default := r.v;
            END FOR;

            LET rs_ss RESULTSET := (
                SHOW PARAMETERS LIKE 'CORTEX_CODE_SNOWSIGHT_DAILY_EST_CREDIT_LIMIT_PER_USER' IN ACCOUNT
            );
            LET df_ss RESULTSET := (SELECT "value"::FLOAT AS v FROM TABLE(RESULT_SCAN(LAST_QUERY_ID())));
            FOR r IN df_ss DO
                ss_default := r.v;
            END FOR;

            IF (cli_default <= 0 AND ss_default <= 0) THEN
                RETURN 'No positive daily limits configured — nothing to check';
            END IF;

            OPEN c_users;
            FOR rec IN c_users DO
                -- Check cooldown: skip if already alerted recently
                LET recent_count INT := (
                    SELECT COUNT(*)
                    FROM {governance_schema}.CORTEX_CODE_LIMIT_ALERTS
                    WHERE user_name = rec.user_name
                      AND surface   = rec.surface
                      AND alerted_at >= DATEADD('hour', -1 * :cooldown_hours, CURRENT_TIMESTAMP)
                );
                IF (recent_count = 0) THEN
                    -- Log the alert
                    INSERT INTO {governance_schema}.CORTEX_CODE_LIMIT_ALERTS
                        (user_name, surface, usage_credits, limit_credits, pct_used)
                    VALUES (rec.user_name, rec.surface, rec.usage_credits,
                            rec.limit_credits, rec.pct_used);

                    -- Send email
                    CALL SYSTEM$SEND_EMAIL(
                        :notif_integration,
                        :notif_email,
                        'Cortex Code limit alert: ' || rec.user_name || ' at ' || rec.pct_used || '% (' || rec.surface || ')',
                        'User ' || rec.user_name || ' has used ' || rec.usage_credits ||
                        ' of ' || rec.limit_credits || ' credits (' || rec.pct_used ||
                        '%) on the ' || rec.surface || ' surface in the last 24 hours.' ||
                        CHR(10) || CHR(10) ||
                        'Threshold: ' || :alert_threshold || '%' || CHR(10) ||
                        'Action: The user will be blocked when they reach 100%. ' ||
                        'Consider increasing their limit or reaching out.'
                    );

                    alert_count := alert_count + 1;
                END IF;
            END FOR;
            CLOSE c_users;

            RETURN alert_count || ' alert(s) sent';
        END;
        $$
    """).collect()
    print("✓ Procedure CORTEX_CODE_LIMIT_ALERT_CHECK created")

    # 3. Serverless task
    session.sql(f"""
        CREATE OR REPLACE TASK {governance_schema}.CORTEX_CODE_LIMIT_ALERT_TASK
            SCHEDULE = '{task_schedule_minutes} MINUTE'
            USER_TASK_MANAGED_INITIAL_WAREHOUSE_SIZE = 'XSMALL'
            COMMENT = 'Checks rolling 24h Cortex Code usage against per-user daily limits and sends alerts'
        AS
            CALL {governance_schema}.CORTEX_CODE_LIMIT_ALERT_CHECK()
    """).collect()
    print(f"✓ Task CORTEX_CODE_LIMIT_ALERT_TASK created (every {task_schedule_minutes} min)")

    session.sql(f"ALTER TASK {governance_schema}.CORTEX_CODE_LIMIT_ALERT_TASK RESUME").collect()
    print("✓ Task resumed — alerts are now active")

except Exception as e:
    print(f"Deployment failed: {e}")

### Alert Status & History
Shows the task run state and recent alerts. Run periodically to confirm the system is working.

In [ ]:
print("=== Task Status ===\n")
try:
    task_info = session.sql(f"""
        SELECT NAME, STATE, SCHEDULE,
               LAST_COMMITTED_ON, LAST_SUSPENDED_ON
        FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
            TASK_NAME => 'CORTEX_CODE_LIMIT_ALERT_TASK',
            SCHEDULED_TIME_RANGE_START => DATEADD('hour', -24, CURRENT_TIMESTAMP)
        ))
        QUALIFY ROW_NUMBER() OVER (ORDER BY SCHEDULED_TIME DESC) = 1
    """).to_pandas()
    if task_info.empty:
        print("No task runs in the last 24 hours (may still be waiting for first schedule).")
    else:
        print(task_info.to_string(index=False))
except Exception:
    try:
        task_show = session.sql(
            f"SHOW TASKS LIKE 'CORTEX_CODE_LIMIT_ALERT_TASK' IN SCHEMA {governance_schema}"
        ).to_pandas()
        if not task_show.empty:
            print(task_show[['name', 'state', 'schedule']].to_string(index=False))
        else:
            print("Task not found. Run the deployment cell above first.")
    except Exception as e:
        print(f"Could not query task status: {e}")

print("\n=== Recent Alerts (last 7 days) ===\n")
try:
    alerts = session.sql(f"""
        SELECT user_name, surface, usage_credits, limit_credits,
               pct_used, alerted_at
        FROM {governance_schema}.CORTEX_CODE_LIMIT_ALERTS
        WHERE alerted_at >= DATEADD('day', -7, CURRENT_TIMESTAMP)
        ORDER BY alerted_at DESC
        LIMIT 25
    """).to_pandas()
    if alerts.empty:
        print("No alerts in the last 7 days.")
    else:
        print(alerts.to_string(index=False))
except Exception as e:
    print(f"Could not query alerts (table may not exist yet): {e}")

print("\n=== Current Custom Budgets ===\n")
try:
    budgets = session.sql("SHOW SNOWFLAKE.CORE.BUDGET INSTANCES IN ACCOUNT").to_pandas()
    if budgets.empty:
        print("No custom budgets configured.")
    else:
        print(budgets.to_string(index=False))
except Exception as e:
    print(f"Could not query budgets (ACCOUNTADMIN or BUDGET_VIEWER required): {e}")

### Teardown
Uncomment and run to remove all notification objects created by this notebook. Budget notifications and per-user limit parameters are **not** affected — only the custom alert task, procedure, and audit table are removed.

In [ ]:
# Uncomment the lines below to tear down notification objects.

# session.sql(f"ALTER TASK {governance_schema}.CORTEX_CODE_LIMIT_ALERT_TASK SUSPEND").collect()
# session.sql(f"DROP TASK IF EXISTS {governance_schema}.CORTEX_CODE_LIMIT_ALERT_TASK").collect()
# session.sql(f"DROP PROCEDURE IF EXISTS {governance_schema}.CORTEX_CODE_LIMIT_ALERT_CHECK()").collect()
# session.sql(f"DROP TABLE IF EXISTS {governance_schema}.CORTEX_CODE_LIMIT_ALERTS").collect()
# session.sql("DROP NOTIFICATION INTEGRATION IF EXISTS cortex_code_budget_email_int").collect()
# session.sql(f"DROP SCHEMA IF EXISTS {governance_schema}").collect()
# print("Teardown complete.")